In [0]:
df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("encoding", "ISO-8859-1")
    .csv("/Volumes/retail_project_dbricks/default/retail_volume/online_retail_II.csv")
)

display(df.limit(10))

In [0]:
print("Rows:", df.count())

In [0]:
df.printSchema()

In [0]:
df_clean = (
    df.withColumnRenamed("Customer ID", "CustomerID")
)

df_clean.printSchema()

In [0]:
(
    df_clean.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("retail_project_dbricks.default.bronze_retail_raw")
)

In [0]:
spark.sql("""
SELECT COUNT(*)
FROM retail_project_dbricks.default.bronze_retail_raw
""").show()

In [0]:
from pyspark.sql.functions import col

df_silver = (
    spark.table("retail_project_dbricks.default.bronze_retail_raw")
    .dropna(subset=["Invoice", "StockCode", "CustomerID"])
    .filter(col("Quantity") > 0)
    .filter(col("Price") > 0)
    .withColumn("Revenue", col("Quantity") * col("Price"))
)